# 🚢 تجارت‌یار — اجرای سامانه روی Google Colabاین دفترچه، سامانه **تجارت‌یار** (React + Express + Vite) را روی Colab نصب، build و اجرا می‌کند و یک **لینک عمومی موقت** در اختیار شما می‌گذارد.**مراحل (به ترتیب اجرا کنید):**1. نصب Node.js ۲۰2. دریافت کد از GitHub + نصب وابستگی‌ها + build3. اجرای سرور تولید (UI + API روی پورت ۳۰۰۰)4. ساخت تونل عمومی (cloudflared) و دریافت لینک> لینک نهایی در خروجی «سلول آخر» چاپ می‌شود. هر سلول را با `Ctrl+Enter` اجرا کنید.

In [ ]:
# ۱) نصب Node.js نسخه ۲۰curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1apt-get install -y nodejs > /dev/null 2>&1echo "Node: $(node -v) — npm: $(npm -v)"

In [ ]:
# ۲) دریافت کد از GitHub، نصب وابستگی‌ها و buildcd /contentrm -rf Tejaratyarrgit clone --depth 1 --branch arena/01a04f8e-tejaratyarr https://github.com/Setayesh-Jafari/Tejaratyarr.gitcd Tejaratyarrnpm installnpm run buildecho build-ok

In [ ]:
# ۳) اجرای سرور تولید (UI + API روی پورت 3000)cd /content/TejaratyarrNODE_ENV=production nohup node dist/server.cjs > /content/server.log 2>&1 &sleep 4curl -s http://localhost:3000/api/health && echo "" || (echo '--- server.log ---' && cat /content/server.log)

In [ ]:
# ۴) ساخت تونل عمومی موقت با cloudflared (بدون نیاز به توکن)cd /content/Tejaratyarrif [ ! -f cloudflared ]; then  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared  chmod +x cloudflaredfinohup ./cloudflared tunnel --url http://localhost:3000 > /content/cloudflared.log 2>&1 &echo tunnel-starting

In [ ]:
import time, reurl = Nonefor _ in range(90):    try:        log = open('/content/cloudflared.log').read()        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)        if m:            url = m.group(0)            break    except Exception:        pass    time.sleep(2)if url:    print('OK link:')    print(url)else:    print('link not ready, cloudflared log:')    try:        print(open('/content/cloudflared.log').read()[-1500:])    except Exception as e:        print(e)

## 📌 نکات- لینک `trycloudflare` **موقت** است و تا زمانی که Colab روشن بماند کار می‌کند.- داده‌ها در Colab موقتی است (با قطع جلسه از بین می‌رود) — برای استفاده‌ی واقعی روی سرور خودتان اجرا کنید.- برای فعال‌سازی تحلیل هوش مصنوعی Gemini، متغیر محیطی `GEMINI_API_KEY` را قبل از اجرای سرور تنظیم کنید.- اجرای محلی:  - حالت توسعه: `npm install` سپس `npm run dev` (پورت ۳۰۰۰)  - حالت تولید: `npm run build` سپس `NODE_ENV=production node dist/server.cjs`